# Amazon Bedrock AgentCore Policy - Getting Started Demo

## Overview

Welcome to the Amazon Bedrock AgentCore Policy hands-on demo! This notebook will guide you through the complete workflow of setting up and testing policy-based deterministic controls for AI agent-tool interactions.

### What is AgentCore Policy?

Amazon Bedrock AgentCore Policy enables developers to define and enforce security controls for AI agent interactions with tools by creating a protective boundary ("safety box") around agent operations. AI agents can dynamically adapt to solve complex problems, but this flexibility introduces security challenges:

- **Data Leakage**: Agents may inadvertently expose private information
- **Business Rule Violations**: Agents might misinterpret or bypass business rules
- **Authority Overreach**: Agents could act outside their intended scope

Policy intercepts inbound agent traffic through AgentCore Gateways and evaluates each request against defined policies before allowing tool access.

### Key Benefits

✅ **Declarative Security**: Define policies using Cedar language, not code  
✅ **Runtime Enforcement**: Policies are evaluated in real-time  
✅ **Fine-Grained Control**: From coarse restrictions to detailed access control/ authroization
✅ **Separation of Concerns**: Security logic lives outside agent code  
✅ **Enterprise Scale**: Deploy autonomous agents safely in production  

---

## Demo Architecture

```
┌─────────────┐
│   AI Agent  │
└──────┬──────┘
       │
       │ Tool Call Request
       ▼
┌─────────────────────┐
│  AgentCore Gateway  │
│  + OAuth Auth       │
└──────┬──────────────┘
       │
       │ Policy Check
       ▼
┌─────────────────────┐
│   Policy Engine     │
│   (Cedar Policies)  │
└──────┬──────────────┘
       │
       │ ALLOW / DENY
       ▼
┌─────────────────────--┐
│   Gateway Targets     │
│                       │    
└─────────────────────--┘
```

---

## What You'll Learn

In this demo, you will set up a agent with tools to help perform insurance underwriting:

1. **Setup Infrastructure**: Create a Gateway with Lambda targets for creating insurance application, invoking a risk model and approving insurance claims
2. **Create Policy Engine**: Initialize a policy engine for your gateway
3. **Define Policies**: Write Cedar policies to control access
4. **Test Enforcement**: Verify policies work with real agent requests
5. **Understand Results**: Interpret ALLOW and DENY scenarios

---

## Prerequisites

Before starting, ensure you have:

- ✅ AWS CLI configured with appropriate credentials
- ✅ Python 3.10+ with boto3 installed
- ✅ `bedrock_agentcore_starter_toolkit` package installed
- ✅ Access to AWS Lambda (for target functions)

---

## Demo Scenario: Insurance Underwriting Processing

We'll implement a **insurance underwriting processing system** with policy controls:

- **Tools**: 
       - `ApplicationTool` - Creates a insurance application for a region and coverage_amount
       - `RiskModelTool` - Invokes external risk scoring model with governance controls, when provided API classification and data governance approval
       - `ApprovalTool` - Approve high-value or high-risk underwriting decisions, parameters claim_amount and risk_level


Let's get started! 🚀

---

# Step 0: Environment Setup

First, let's verify our environment and import necessary libraries.

In [1]:
# Install requirements
%pip install -r requirements.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 67.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 69.5 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 66.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 914.9/914.9 kB 10.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 59.5 MB/s  0:00:00
  Attempting uninstall: jupyterlab90m╺━━━━━━━━━━━━━━━━━━━  5/10 [ipywidgets]
    Found existing installation: jupyterlab 4.6.1━━━━━━━━━━━━━  5/10 [ipywidgets]
    Uninstalling jupyterlab-4.6.1:━━━━━━━━━━━━━━━━━━━  5/10 [ipywidgets]
      Successfully uninstalled jupyterlab-4.6.1━━━━━━━━━━━━━━━  5/10 [ipywidgets]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10/10 [jupyter]8/10 [notebook]b]

[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [ ]:
# Restart the kernel to adopt the updates
import IPython

IPython.Application.instance().kernel.do_shutdown(True)

{'status': 'ok', 'restart': True}

: 

In [1]:
# Import required libraries
import sys
from pathlib import Path
import boto3
import json
import logging

In [2]:
# Add the scripts directory to Python path
scripts_dir = Path.cwd() / "scripts"
if str(scripts_dir) not in sys.path:
    sys.path.insert(0, str(scripts_dir))

# Prompt for region
session = boto3.Session()
region = session.region_name
if not region:
    region = input("Enter AWS region (e.g., us-east-1, us-west-2): ").strip()
    if not region:
        raise ValueError("AWS region is required")

print(f"Region: {region}")

# Verify AWS credentials
try:
    sts = session.client("sts", region_name=region)
    identity = sts.get_caller_identity()
    print("✅ AWS Credentials Verified")
    print(f"   Account: {identity['Account']}")
    print(f"   User/Role: {identity['Arn']}")
except Exception as e:
    print(f"❌ AWS Credentials Error: {e}")
    print("   Please configure AWS CLI with: aws configure")

Region: us-east-1
✅ AWS Credentials Verified
   Account: 402020382644
   User/Role: arn:aws:iam::402020382644:user/palm


---

# Step 1: Create Target Functions for AgentCore Gateway

Before setting up the gateway, we need functions that will serve as our tool targets for the Agent.

## What is a Lambda Target?

A Lambda target is a backend function that the AI agent can invoke through the gateway. In our case, we are setting up 3 lambda functions, for ApplicationTool, RiskModelTool and ApprovalTool to support the agent perform insurance underwriting tasks. 

### Create via CLI (run the cell below)
Running the following script will deploy 3 Lambdas in your AWS account:

1. Application Tool: Simplified Application Creation (Mocked Up for Demo purpose)
 Creates insurance applications with applicant region and coverage amount
 Parameters:
 - applicant_region: Customer's geographic region
 - coverage_amount: Requested insurance coverage amount

2. Risk Model Tool: Simplified Risk Model Access (Mocked Up for Demo purpose)
 Invokes risk scoring model and returns assessment
 Parameters:
 - API_classification: API classification (public, internal, restricted)
 - data_governance_approval: Whether data governance has approved model usage

3. Approval Tool: - Insurance Approval Process (Mocked Up for Demo purpose)
 Approves underwriting decisions and claim amounts
 Parameters:
 - claim_amount: Insurance claim/coverage amount
 - risk_level: Risk level assessment (low, medium, high, critical)

In [3]:
%run scripts/lambda-target-setup/deploy_lambdas.py --region $region

🚀 Deploying Lambda Functions


Region: us-east-1

🔐 No role provided, setting up IAM role...
   ✅ Using existing IAM role: AgentCoreLambdaExecutionRole

📦 Deploying ApplicationTool...
   ℹ️  Function exists, updating code...
   ✅ Code updated
   ARN: arn:aws:lambda:us-east-1:402020382644:function:ApplicationTool

📦 Deploying ApprovalTool...
   ℹ️  Function exists, updating code...
   ✅ Code updated
   ARN: arn:aws:lambda:us-east-1:402020382644:function:ApprovalTool

📦 Deploying RiskModelTool...
   ℹ️  Function exists, updating code...
   ✅ Code updated
   ARN: arn:aws:lambda:us-east-1:402020382644:function:RiskModelTool


💾 Configuration saved to: /Users/pathipat/Work/Learning/Yip-aws-strands&agentcore/agentcore-samples/06-workshops/08-AgentCore-policy/01-Getting-Started/config.json

✅ Deployment complete! 3/3 functions deployed.

Lambda ARNs have been saved to config.json



---

# Step 2: Setup AgentCore Gateway

Now we'll create an AgentCore Gateway with OAuth authentication and attach our Lambda functions as targets.

## What Gets Created?

1. **OAuth Authorization Server**: Cognito-based OAuth for authentication
2. **AgentCore Gateway**: Main gateway with MCP protocol support
3. **Lambda Target**: Your Lambda functions attached with schema definition
4. **Configuration File**: All connection details saved for later use

## Gateway Configuration

The gateway will be configured with:
- **Protocol**: MCP (Model Context Protocol)
- **Authentication**: OAuth 2.0 via Cognito
- **Target**: ApplicationTool, RiskModelTool, ApprovalTool Lambda functions - Target schema will be provided to the Gateway

In [4]:
# Run the gateway setup script
print("🚀 Setting up AgentCore Gateway...\n")
print("This will:")
print("  1. Create OAuth authorization server (Cognito)")
print("  2. Create AgentCore Gateway")
print("  3. Attach Lambda as target")
print("  4. Save configuration to config.json")
print("\n" + "=" * 60)

# Run the gateway setup script
%run scripts/setup_gateway.py

🚀 Setting up AgentCore Gateway...

This will:
  1. Create OAuth authorization server (Cognito)
  2. Create AgentCore Gateway
  3. Attach Lambda as target
  4. Save configuration to config.json

🚀 Setting up AgentCore Gateway for Insurance Underwriting...

📦 Loading configuration...
Region: us-east-1



2026-08-07 13:46:45,582 - bedrock_agentcore.gateway - INFO - Starting EZ Auth setup: Creating Cognito resources...


✅ Found Lambda functions:
   • ApplicationTool: arn:aws:lambda:us-east-1:402020382644:function:ApplicationTool
   • ApprovalTool: arn:aws:lambda:us-east-1:402020382644:function:ApprovalTool
   • RiskModelTool: arn:aws:lambda:us-east-1:402020382644:function:RiskModelTool

� Iniutializing AgentCore client...

📝 Step 1: Creating OAuth authorization server...


2026-08-07 13:46:47,475 - bedrock_agentcore.gateway - INFO -   ✓ Created User Pool: us-east-1_aucbNP7AE
2026-08-07 13:46:48,465 - bedrock_agentcore.gateway - INFO -   ✓ Created domain: agentcore-0c0c4be3
2026-08-07 13:46:48,466 - bedrock_agentcore.gateway - INFO -   ⏳ Waiting for domain to be available...
2026-08-07 13:46:48,804 - bedrock_agentcore.gateway - INFO -   ✓ Domain is active
2026-08-07 13:46:49,317 - bedrock_agentcore.gateway - INFO -   ✓ Created resource server: InsuranceUnderwritingGateway
2026-08-07 13:46:49,838 - bedrock_agentcore.gateway - INFO -   ✓ Created client: 2hgj9ik53blr56romvmju48a12
2026-08-07 13:46:49,839 - bedrock_agentcore.gateway - INFO -   ⏳ Waiting for DNS propagation of domain: agentcore-0c0c4be3.auth.us-east-1.amazoncognito.com
2026-08-07 13:47:49,843 - bedrock_agentcore.gateway - INFO - ✓ EZ Auth setup complete!
2026-08-07 13:47:49,852 - bedrock_agentcore.gateway - INFO - Role not provided, creating an execution role to use


✅ Authorization server created

📝 Step 2: Creating AgentCore Gateway...


...

   ✅ Successfully added ApplicationTool target

   🔧 Adding RiskModelTool target...


2026-08-07 13:48:36,751 - bedrock_agentcore.gateway - INFO - ✓ Added target successfully (ID: FQUA431XJ0)
2026-08-07 13:48:36,751 - bedrock_agentcore.gateway - INFO -   Waiting for target to be ready...
2026-08-07 13:48:39,533 - bedrock_agentcore.gateway - INFO - 
✅Target is ready
2026-08-07 13:48:39,534 - bedrock_agentcore.gateway - INFO - Creating Target
2026-08-07 13:48:39,534 - bedrock_agentcore.gateway - INFO - {'gatewayIdentifier': 'gw-insurance-underwriting-htmcvhhnpa', 'name': 'ApprovalToolTarget', 'targetConfiguration': {'mcp': {'lambda': {'lambdaArn': 'arn:aws:lambda:us-east-1:402020382644:function:ApprovalTool', 'toolSchema': {'inlinePayload': [{'name': 'approve_underwriting', 'description': 'Approve high-value or high-risk underwriting decisions', 'inputSchema': {'type': 'object', 'description': 'Input parameters for underwriting approval', 'properties': {'claim_amount': {'type': 'integer', 'description': 'Insurance claim/coverage amount'}, 'risk_level': {'type': 'string', 

   ✅ Successfully added RiskModelTool target

   🔧 Adding ApprovalTool target...


2026-08-07 13:48:40,128 - bedrock_agentcore.gateway - INFO - ✓ Added target successfully (ID: DVVPGHB124)
2026-08-07 13:48:40,128 - bedrock_agentcore.gateway - INFO -   Waiting for target to be ready...
2026-08-07 13:48:45,321 - bedrock_agentcore.gateway - INFO - 
✅Target is ready


   ✅ Successfully added ApprovalTool target

📝 Step 4: Updating config.json with gateway information...

✅ GATEWAY SETUP COMPLETE!
Gateway Name: GW-Insurance-Underwriting
Gateway URL: https://gw-insurance-underwriting-htmcvhhnpa.gateway.bedrock-agentcore.us-east-1.amazonaws.com/mcp
Gateway ID: gw-insurance-underwriting-htmcvhhnpa
Gateway ARN: arn:aws:bedrock-agentcore:us-east-1:402020382644:gateway/gw-insurance-underwriting-htmcvhhnpa

Targets Added: 3
   • ApplicationTool
   • RiskModelTool
   • ApprovalTool

Configuration updated in: /Users/pathipat/Work/Learning/Yip-aws-strands&agentcore/agentcore-samples/06-workshops/08-AgentCore-policy/01-Getting-Started/config.json


### Verify Gateway Configuration

Let's load and verify the gateway configuration that was just created:

In [5]:
# Load gateway configuration
gateway_config_file = "config.json"

with open(gateway_config_file, "r") as f:
    gateway_config = json.load(f)

print("✅ Gateway Configuration Loaded\n")
print("=" * 60)
print(f"Gateway ID:  {gateway_config['gateway']['gateway_id']}")
print(f"Gateway ARN: {gateway_config['gateway']['gateway_arn']}")
print(f"Gateway URL: {gateway_config['gateway']['gateway_url']}")
print(f"Region:      {gateway_config['region']}")
print("\nOAuth Configuration:")
print(f"  Client ID:  {gateway_config['gateway']['client_info']['client_id']}")
print(f"  Token URL:  {gateway_config['gateway']['client_info']['token_endpoint']}")
print("=" * 60)

# Store for later use
GATEWAY_ARN = gateway_config["gateway"]["gateway_arn"]
GATEWAY_ID = gateway_config["gateway"]["gateway_id"]
GATEWAY_URL = gateway_config["gateway"]["gateway_url"]

✅ Gateway Configuration Loaded

Gateway ID:  gw-insurance-underwriting-htmcvhhnpa
Gateway ARN: arn:aws:bedrock-agentcore:us-east-1:402020382644:gateway/gw-insurance-underwriting-htmcvhhnpa
Gateway URL: https://gw-insurance-underwriting-htmcvhhnpa.gateway.bedrock-agentcore.us-east-1.amazonaws.com/mcp
Region:      us-east-1

OAuth Configuration:
  Client ID:  2hgj9ik53blr56romvmju48a12
  Token URL:  https://agentcore-0c0c4be3.auth.us-east-1.amazoncognito.com/oauth2/token


### Run a agent which uses the tools created on the Gateway

In [6]:
# Import the agent session
from scripts.agent_with_tools import AgentSession

# Use the agent within a context manager (this handles setup and cleanup automatically)
with AgentSession() as session:
    # The agent will list all available tools during setup

    # Now you can invoke the agent with different prompts
    response1 = session.invoke("What tools do you have access to?")

    response2 = session.invoke("Create an application for US region with $5M coverage")

    response3 = session.invoke(
        "Invoke the risk model with public API classification and data governance approval set to true"
    )

    response4 = session.invoke("Approve underwriting for $75000 claim with medium risk level")

# The session is automatically cleaned up when exiting the 'with' block
print("=" * 60)
print(f"🚀 The agent has access to the following tools configured on the Gateway: {response1}\n")
print(f"🚀 The agent can create applications without any limits: {response2}\n")
print(f"🚀 The agent can invoke the risk model: {response3}\n")
print(f"🚀 The agent is able to approve the insurance claims: {response4}\n")
print("=" * 60)

📦 Loading configuration...
✅ Configuration loaded
   Gateway: GW-Insurance-Underwriting
   Region: us-east-1

🔑 Authenticating...
✅ Authentication successful

📋 Listing available tools...
✅ Found 4 tool(s):
   • x_amz_bedrock_agentcore_search
   • ApplicationToolTarget___create_application
   • ApprovalToolTarget___approve_underwriting
   • RiskModelToolTarget___invoke_risk_model

🤖 Setting up model: us.amazon.nova-lite-v1:0
✅ Agent ready!

💬 Prompt: What tools do you have access to?

🤔 Thinking...

You have access to the following tools:

1. **x_amz_bedrock_agentcore_search**: A tool that returns a trimmed down list of tools given a context. It requires a query string.
2. **ApplicationToolTarget___create_application**: A tool to create an insurance application with geographic and eligibility validation. It requires the applicant's region and the requested coverage amount.
3. **ApprovalToolTarget___approve_underwriting**: A tool to approve high-value or high-risk underwriting decisions

---

# Step 3: Create Policy Engine and Policies

Now we'll create a Policy Engine with Cedar policies to control access to the tools on the gateway.

## What is a Policy Engine?

A Policy Engine evaluates requests against Cedar policies in real-time. It operates in two modes:
- **LOG_ONLY**: Evaluates but doesn't block (for testing)
- **ENFORCE**: Actively blocks non-compliant requests (for production)

When a Gateway is associated with a Policy Engine, the default action is deny, unless specific policies allow access. An empty policy engine will now allow any tools on the gateway to be accessed. 

### Encryption Keys

When creating a Policy Engine, you can apply a Customer Managed Key (CMK), which is a customer owned and managed key present in the AWS Key Management Service (KMS).  Before creating the Policy Engine, you will be prompted for the ARN of the CMK.  If you do not want to use one, just simply enter no value.

In [7]:
# Get the CMK ARN from the user
user_input = input("Please enter the CMK ARN.  To skip enter a blank value: ")

# If the user entered a value, use it. Otherwise, pass None
cmk_arn = None
if user_input.strip():  # Check if the input is not empty
    cmk_arn = user_input.strip()

### Create Policy Engine

First, we'll create a Policy Engine to hold our Cedar policies:

In [8]:
# Import PolicyClient from AgentCore Starter Toolkit
import boto3

# Create the needed constructs to create the policy engine
client = boto3.client("bedrock-agentcore-control", region_name=region)

# Create a Policy Engine
print("🔧 Creating Policy Engine...")

# Capture the data to send into the client
request = {
    "name": "InsurancePolicyEngine",
    "description": "Policy engine for insurance underwriting governance",
}

# If a CMK was provided, add it to the request to create the policy engine
if cmk_arn:
    request["encryptionKeyArn"] = cmk_arn


# Create the policy engine (reuse if one with same name already exists)
try:
    engine = client.create_policy_engine(**request)
except client.exceptions.ConflictException:
    existing = client.list_policy_engines()
    engine = next(e for e in existing.get("policyEngines", []) if e["name"] == request["name"])

print(f"✓ Policy Engine: {engine['policyEngineId']}\n")

🔧 Creating Policy Engine...
✓ Policy Engine: InsurancePolicyEngine-x8ojpx3bcn



In [9]:
# Save Policy Engine in the configuration file
with open("config.json", "r") as f:
    config = json.load(f)

# Add policy engine information (without removing existing data)
config["policy_engine_id"] = engine["policyEngineId"]
config["policy_engine_arn"] = engine["policyEngineArn"]

# Write back the updated config
with open("config.json", "w") as f:
    json.dump(config, f, indent=2)

print("✅ Policy engine information added to config.json")

✅ Policy engine information added to config.json


In [10]:
# Wait for Policy Engine to become ACTIVE before attaching
import time

_pe_client = boto3.client("bedrock-agentcore-control", region_name=region)
for _ in range(60):
    _pe_status = _pe_client.get_policy_engine(policyEngineId=engine["policyEngineId"]).get("status")
    if _pe_status == "ACTIVE":
        break
    print(f"Policy Engine status: {_pe_status}, waiting...")
    time.sleep(5)
print(f"Policy Engine is {_pe_status}")

# Attach Policy Engine to the Gateway
from bedrock_agentcore_starter_toolkit.operations.gateway.client import GatewayClient

gateway_client = GatewayClient(region_name=region)
gateway_client.logger.setLevel(logging.INFO)

gateway_client.update_gateway_policy_engine(
    gateway_identifier=config["gateway"]["gateway_id"],
    policy_engine_arn=engine["policyEngineArn"],
    mode="ENFORCE",
)
print("✓ Policy Engine attached to Gateway\n")

2026-08-07 14:50:01,076 - bedrock_agentcore.gateway - INFO - Attaching policy engine to gateway
2026-08-07 14:50:01,077 - bedrock_agentcore.gateway - INFO - Updating gateway gw-insurance-underwriting-htmcvhhnpa


Policy Engine is ACTIVE


2026-08-07 14:50:01,984 - bedrock_agentcore.gateway - INFO -   Policy Engine ARN: arn:aws:bedrock-agentcore:us-east-1:402020382644:policy-engine/InsurancePolicyEngine-x8ojpx3bcn
2026-08-07 14:50:01,984 - bedrock_agentcore.gateway - INFO -   Mode: ENFORCE
2026-08-07 14:50:02,706 - bedrock_agentcore.gateway - INFO - ✓ Gateway update initiated
2026-08-07 14:50:02,707 - bedrock_agentcore.gateway - INFO -   Waiting for gateway to be ready...
2026-08-07 14:50:05,385 - bedrock_agentcore.gateway - INFO - ✓ Gateway update complete


✓ Policy Engine attached to Gateway



### Now that the Policy Engine is attached to the Gateway, by default the tools will be blocked, both during the list tools API call as well as when a agent tries to access it through the Gateway

In [11]:
# Import the agent session
from scripts.agent_with_tools import list_available_tools, fetch_access_token

client_info = config["gateway"]["client_info"]

CLIENT_ID = client_info["client_id"]
CLIENT_SECRET = client_info["client_secret"]
TOKEN_URL = client_info["token_endpoint"]
access_token = fetch_access_token(CLIENT_ID, CLIENT_SECRET, TOKEN_URL)

print("=" * 60)
print("🚀 The agent has access to the following tools configured on the Gateway: \n")
print(list_available_tools(config["gateway"]["gateway_url"], access_token))

🚀 The agent has access to the following tools configured on the Gateway: 

[]


### Create Cedar Policy

Now we'll create a Cedar policy that allows insurance applications to be created if the coverage amount is below 1 million:

## Our Policy Rule

```cedar
permit(
  principal,
  action == AgentCore::Action::"ApplicationToolTarget___create_application",
  resource == AgentCore::Gateway::"<gateway-arn>"
) when {
  context.input.coverage_amount <= 1000000
};
```

This means:
- ✅ Create insurance application of $1000000 or less: **ALLOWED**
- ❌ Create insurance application over $1000000: **DENIED**

In [12]:
from bedrock_agentcore_starter_toolkit.operations.policy.client import PolicyClient

# Policy client creation
policy_client = PolicyClient(region_name=region)
policy_client.logger.setLevel(logging.INFO)

# Create Cedar policy
print("\n📝 Creating Cedar Policy...")
print(f"   Policy Engine ID: {engine['policyEngineArn']}")
GATEWAY_ARN = config["gateway"]["gateway_arn"]

# Define the Cedar policy statement
cedar_statement = (
    f"permit(principal, "
    f'action == AgentCore::Action::"ApplicationToolTarget___create_application", '
    f'resource == AgentCore::Gateway::"{GATEWAY_ARN}") '
    f"when {{ context.input.coverage_amount <= 1000000 }};"
)

try:
    policy = policy_client.create_or_get_policy(
        policy_engine_id=engine["policyEngineId"],
        name="create_application_policy",
        description="Allow application creation under $1M",
        definition={"cedar": {"statement": cedar_statement}},
    )
    print(f"✓ Policy: {policy['policyId']}\n")
except Exception as e:
    print(f"⚠️  Policy creation failed: {e}")
    print("   This may be due to Cedar validation findings (e.g., policy too restrictive or schema issues).")
    print("   Retrying with validation_mode='IGNORE_ALL_FINDINGS'...")
    policy = policy_client.create_or_get_policy(
        policy_engine_id=engine["policyEngineId"],
        name="create_application_policy",
        description="Allow application creation under $1M",
        definition={"cedar": {"statement": cedar_statement}},
        validation_mode="IGNORE_ALL_FINDINGS",
    )
    print(f"✓ Policy created with IGNORE_ALL_FINDINGS: {policy['policyId']}\n")

# Save to config
config["policy_id"] = policy["policyId"]
with open("config.json", "w") as f:
    json.dump(config, f, indent=2)

2026-08-07 14:50:24,028 - bedrock_agentcore.policy - INFO - Creating or getting Policy: create_application_policy



📝 Creating Cedar Policy...
   Policy Engine ID: arn:aws:bedrock-agentcore:us-east-1:402020382644:policy-engine/InsurancePolicyEngine-x8ojpx3bcn


2026-08-07 14:50:24,995 - bedrock_agentcore.policy - INFO - Creating Policy: create_application_policy
2026-08-07 14:50:25,472 - bedrock_agentcore.policy - INFO - ✓ Policy creation initiated: arn:aws:bedrock-agentcore:us-east-1:402020382644:policy-engine/InsurancePolicyEngine-x8ojpx3bcn/policy/create_application_policy-f9c_rxv800
2026-08-07 14:50:25,472 - bedrock_agentcore.policy - INFO - Waiting for Policy to be active...
2026-08-07 14:50:30,469 - bedrock_agentcore.policy - INFO - ✓ Policy is active


✓ Policy: create_application_policy-f9c_rxv800



---

# Step 4: Test Policy Enforcement with AI Agent

Now for the exciting part - let's test our policy with a real AI agent!
You will also now notice that due to the policy we have attached for the create_application tool, this tool can now be listed by the gateway and invoked by the agent

## Test Scenarios

We'll test two scenarios:

### Test 1: ALLOWED Scenario ✅
- **Request**: Create a application with a coverage amount of a $750,000
- **Expected**: Policy allows, Lambda executes, application created
- **Reason**: $750K <= $1M (within policy limit)


In [13]:
# Use the agent within a context manager (this handles setup and cleanup automatically)
with AgentSession() as session:
    # The agent will list all available tools during setup

    # Now you can invoke the agent with different prompts
    response1 = session.invoke("What tools do you have access to?")

    response2 = session.invoke("Create an application for US region with $750,000 coverage")

📦 Loading configuration...
✅ Configuration loaded
   Gateway: GW-Insurance-Underwriting
   Region: us-east-1

🔑 Authenticating...
✅ Authentication successful

📋 Listing available tools...
✅ Found 1 tool(s):
   • ApplicationToolTarget___create_application

🤖 Setting up model: us.amazon.nova-lite-v1:0
✅ Agent ready!

💬 Prompt: What tools do you have access to?

🤔 Thinking...

<thinking>The User is asking about the available tools. This is straightforward information that can be provided directly.</thinking>
You have access to the following tool:
- **ApplicationToolTarget___create_application**: Create insurance application with geographic and eligibility validation.

Here are the parameters for the tool:
- **applicant_region**: Customer's geographic region (US, CA, UK, EU, APAC, etc.)
- **coverage_amount**: Requested insurance coverage amount

Both `applicant_region` and `coverage_amount` are required parameters.🤖 Agent: [{'text': "<thinking>The User is asking about the available tools. 

### Test 2: DENIED Scenario ❌
- **Request**: Create a application with a coverage amount of a $1.5M
- **Expected**: Policy blocks, Lambda never executes
- **Reason**: $1.5M > $1M (exceeds policy limit)

In [14]:
with AgentSession() as session:
    # The agent will list all available tools during setup

    response2 = session.invoke("Create an application for US region with $1.5M coverage")

📦 Loading configuration...
✅ Configuration loaded
   Gateway: GW-Insurance-Underwriting
   Region: us-east-1

🔑 Authenticating...
✅ Authentication successful

📋 Listing available tools...
✅ Found 1 tool(s):
   • ApplicationToolTarget___create_application

🤖 Setting up model: us.amazon.nova-lite-v1:0
✅ Agent ready!

💬 Prompt: Create an application for US region with $1.5M coverage

🤔 Thinking...

<thinking>The user wants to create an insurance application for the US region with a coverage amount of $1.5M. I need to use the 'create_application' tool for this. The required parameters are 'applicant_region' and 'coverage_amount', and the user has provided both. I will proceed to use the tool.</thinking>

Tool #1: ApplicationToolTarget___create_application


tool execution failed
Traceback (most recent call last):
  File "/Users/pathipat/Work/Learning/Yip-aws-strands&agentcore/agentcore-samples/.venv/lib/python3.14/site-packages/strands/tools/mcp/mcp_client.py", line 927, in call_tool_async
    call_tool_result: MCPCallToolResult = await asyncio.wrap_future(future)
                                          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/pathipat/Work/Learning/Yip-aws-strands&agentcore/agentcore-samples/.venv/lib/python3.14/site-packages/strands/tools/mcp/mcp_client.py", line 1327, in run_async
    return await invoke_event
           ^^^^^^^^^^^^^^^^^^
  File "/Users/pathipat/Work/Learning/Yip-aws-strands&agentcore/agentcore-samples/.venv/lib/python3.14/site-packages/strands/tools/mcp/mcp_client.py", line 812, in _call_tool_direct
    return await session.call_tool(
           ^^^^^^^^^^^^^^^^^^^^^^^^
        name, arguments, read_timeout_seconds, progress_callback=effective_callback, meta=meta
        ^^^^^^^^^^^^^^^^^^^

<thinking>The tool execution failed due to a policy enforcement error. I will inform the user about the failure and suggest that I cannot proceed with the tool at the moment. I will also ask the user if they would like to try another region or coverage amount.</thinking>

The tool execution failed due to policy enforcement. I'm unable to proceed with the tool at the moment. Would you like to try another region or coverage amount?🤖 Agent: [{'text': "<thinking>The tool execution failed due to a policy enforcement error. I will inform the user about the failure and suggest that I cannot proceed with the tool at the moment. I will also ask the user if they would like to try another region or coverage amount.</thinking>\n\nThe tool execution failed due to policy enforcement. I'm unable to proceed with the tool at the moment. Would you like to try another region or coverage amount?"}]

✅ Agent session closed


## Clean Up

<div style="background-color: #d1ecf1; border-left: 4px solid #0c5460; padding: 10px; margin: 10px 0; color: #000;">
    <strong style="color: #000;">ℹ️ Note:</strong> 02-Natural-Language-Policy-Authoring/NL-Authoring-Policy.ipynb will reuse the Gateway set up from this Demo. You can skip the Cleanup step, and perform it after testing NL2Cedar functionality in the second lab. 
</div>


To clean up the resources of policy engines and policies, it is done in the following order:
1. Delete the association of the policy engine on the gateway by using the update_gateway CLI and passing in a empty policy engine

2. Delete all the policies in the policy engine

3. Delete the policy engine

In [15]:
from bedrock_agentcore_starter_toolkit.operations.gateway.client import GatewayClient
from bedrock_agentcore_starter_toolkit.operations.policy.client import PolicyClient

with open("config.json", "r") as f:
    config = json.load(f)

# Clean up Policy Engine first
print("🧹 Cleaning up Policy Engine...")
policy_client = PolicyClient(region_name=config["region"])
policy_client.cleanup_policy_engine(config["policy_engine_id"])
print("✓ Policy Engine cleaned up\n")

# Then clean up Gateway
print("🧹 Cleaning up Gateway...")
gateway_client = GatewayClient(region_name=config["region"])
gateway_client.cleanup_gateway(config["gateway"]["gateway_id"], config["gateway"]["client_info"])
print("✅ Cleanup complete!")

2026-08-07 14:51:40,300 - bedrock_agentcore.policy - INFO - 🧹 Cleaning up Policy Engine: InsurancePolicyEngine-x8ojpx3bcn


🧹 Cleaning up Policy Engine...


2026-08-07 14:51:41,313 - bedrock_agentcore.policy - INFO - Found 1 policies to delete
2026-08-07 14:51:41,315 - bedrock_agentcore.policy - INFO -   • Deleting policy: create_application_policy
2026-08-07 14:51:41,315 - bedrock_agentcore.policy - INFO - Deleting Policy: create_application_policy-f9c_rxv800
2026-08-07 14:51:41,695 - bedrock_agentcore.policy - INFO - ✓ Policy deletion initiated: create_application_policy-f9c_rxv800
2026-08-07 14:51:41,696 - bedrock_agentcore.policy - INFO -     ✓ Policy deletion initiated: create_application_policy
2026-08-07 14:51:46,815 - bedrock_agentcore.policy - INFO -     ✓ Policy deleted
2026-08-07 14:51:46,816 - bedrock_agentcore.policy - INFO -   • Deleting policy engine: InsurancePolicyEngine-x8ojpx3bcn
2026-08-07 14:51:46,816 - bedrock_agentcore.policy - INFO - Deleting Policy Engine: InsurancePolicyEngine-x8ojpx3bcn
2026-08-07 14:51:47,213 - bedrock_agentcore.policy - INFO - ✓ Policy Engine deletion initiated: InsurancePolicyEngine-x8ojpx3bcn

✓ Policy Engine cleaned up

🧹 Cleaning up Gateway...


2026-08-07 14:51:48,312 - bedrock_agentcore.gateway - INFO -     Found 3 targets to delete
2026-08-07 14:51:48,313 - bedrock_agentcore.gateway - INFO -   • Deleting target: DVVPGHB124
2026-08-07 14:51:48,686 - bedrock_agentcore.gateway - INFO -     ✓ Target deletion initiated: DVVPGHB124
2026-08-07 14:51:53,692 - bedrock_agentcore.gateway - INFO -   • Deleting target: FQUA431XJ0
2026-08-07 14:51:54,079 - bedrock_agentcore.gateway - INFO -     ✓ Target deletion initiated: FQUA431XJ0
2026-08-07 14:51:59,085 - bedrock_agentcore.gateway - INFO -   • Deleting target: IV7XR6SPYF
2026-08-07 14:51:59,470 - bedrock_agentcore.gateway - INFO -     ✓ Target deletion initiated: IV7XR6SPYF
2026-08-07 14:52:04,476 - bedrock_agentcore.gateway - INFO -   • Verifying targets deletion...
2026-08-07 14:52:09,828 - bedrock_agentcore.gateway - INFO -     ✓ All targets deleted
2026-08-07 14:52:09,829 - bedrock_agentcore.gateway - INFO -   • Deleting gateway: gw-insurance-underwriting-htmcvhhnpa
2026-08-07 14

✅ Cleanup complete!
